# Through-Sample Interface Study

Stage D notebook. The input is an air-side `surface_plane` field. The sample branch uses a planar interface, explicit in-medium coordinates, and labels the correction branch as `ideal_numerical_correction` unless a hardware route implements it.

In [1]:
from dataclasses import replace
from pathlib import Path

import pandas as pd

import bessel_twin_core as bt
from vbb_study import setup_study, vbb_regime, vbb_sample_study, viz_fields
from vbb_study.publication import lab_realism as lab_schema

PATHS = setup_study.bootstrap(Path.cwd())
PRESET = "fast"
RUN_ID = PATHS.get("run_id") or None
out_fig = PATHS["figures"] / "stage_d"
out_csv = PATHS["csv"] / "stage_d"
out_fig.mkdir(parents=True, exist_ok=True)
out_csv.mkdir(parents=True, exist_ok=True)

base0 = bt.default_config(PRESET)
notebook_grid = replace(base0.grid, axial_points=25, coarse_scan_points=17, crop_pixels=160)
base0 = replace(base0, grid=notebook_grid)

def _route_method(method):
    return "physical_axicon" if method == "physical" else "holographic_axicon"

def _route_hardware_status(method, variant):
    if variant == "ideal":
        return "simulation_only"
    return "future_hardware_required" if method == "physical" else "current_lab_realizable"


## Editable Notebook Controls

<!-- STAGE88: editable controls -->

This cell exposes the intended user-editable controls for exploratory runs. The locked stage logic below is preserved: changing these controls is for local investigation unless the notebook explicitly wires a value into a regenerated canonical output. Keep QA, caveats, and fail/marginal labels visible. For fast beam-to-sample exploration use the quicklook notebook; for publication-grade outputs use the locked stage runner.


In [2]:
# STAGE88: visible editable controls for exploratory notebook use.
# Edit NOTEBOOK_CONTROLS below and re-run this cell to apply parameter
# overrides to `base0` before running any study cell below.
from vbb_study.publication import notebook_controls as nb_controls
from vbb_study.config import um as _um

NOTEBOOK_CONTROLS = nb_controls.make_notebook_controls(
    stage='lab_realism',
    # ── edit these to override the base configuration ───────────────────────
    ell=3,
    target_core_diameter_um=3.0,
    target_bessel_length_um=150.0,
    objective_NA=0.45,
    blaze_period_px=20,
)

# Wire control parameters into `base0` so downstream cells use them.
_p = NOTEBOOK_CONTROLS.parameters or {}
if "ell" in _p:
    base0 = replace(base0, target=replace(base0.target, ell=int(_p["ell"])))
if "target_core_diameter_um" in _p:
    base0 = replace(base0, target=replace(base0.target, target_core_diameter_m=float(_p["target_core_diameter_um"]) * _um))
if "target_bessel_length_um" in _p:
    base0 = replace(base0, target=replace(base0.target, target_bessel_length_m=float(_p["target_bessel_length_um"]) * _um))
if "objective_NA" in _p:
    base0 = replace(base0, objective=replace(base0.objective, NA=float(_p["objective_NA"])))
if "blaze_period_px" in _p:
    base0 = replace(base0, slm=replace(base0.slm, blaze_period_px=int(_p["blaze_period_px"])))

try:
    display(nb_controls.describe_controls(NOTEBOOK_CONTROLS))
except NameError:
    print(nb_controls.describe_controls(NOTEBOOK_CONTROLS).to_string(index=False))


,control,value
0,stage,lab_realism
1,run_mode,balanced
2,save_outputs,False
3,use_canonical_outputs,True
4,allow_publication_export,False
5,notes,Edit for exploration; keep QA labels/caveats v...
6,ell,3
7,target_core_diameter_um,3.0
8,target_bessel_length_um,150.0
9,objective_NA,0.45


In [3]:
# Interactive quicklook — adjust sliders and click "Update plots".
# Runs a fast preview only; nothing is saved.
from vbb_study.publication import notebook_widgets as nbw

_panel = nbw.interactive_quicklook(base0, method='holographic', preset='fast')
display(_panel)


In [4]:
def configured_case(method, regime, variant):
    cfg = replace(base0, generation_method=method)
    cfg = vbb_regime.config_for_regime(cfg, regime)
    if method == "physical":
        levels = 256 if variant == "lab" else None
        cfg = replace(cfg, physical_axicon=replace(cfg.physical_axicon, slm2_stroke_levels=levels, slm2_conjugate_mode="preserve_vortex"))
        path = "ideal"
    else:
        path = "realistic" if variant == "lab" else "ideal"
    return cfg, path

def winding_fields(beam, method, cfg):
    surface = beam["surface_field"]
    design = beam["design"]
    measured = viz_fields.phase_winding(
        surface.Ex, surface.grid, float(design.vortex_main_ring_radius_m)
    )
    error = abs(measured - int(design.ell))
    if method == "physical" and error >= 0.1:
        raise RuntimeError(
            f"physical winding {measured:.6g} does not match requested ell={design.ell}"
        )
    return {
        "requested_vortex_charge": int(design.ell),
        "measured_winding": float(measured),
        "winding_error": float(error),
        "winding_pass": bool(error < 0.1),
        "slm2_conjugate_mode": (
            str(cfg.physical_axicon.slm2_conjugate_mode) if method == "physical" else "not_applicable"
        ),
        "vortex_removal_acknowledged": (
            bool(cfg.physical_axicon.allow_vortex_removal) if method == "physical" else False
        ),
        "propagation_power_drift_fraction": beam["metrics"].get("propagation_power_drift_fraction"),
        "propagation_power_label": beam["metrics"].get("propagation_power_label"),
        "quantitative_metrics_valid": beam["metrics"].get("quantitative_metrics_valid"),
        "quantitative_metrics_invalid_reason": beam["metrics"].get("quantitative_metrics_invalid_reason"),
    }

def row_for(label, regime, method, variant, beam, uncorrected, corrected, winding):
    cm = corrected.metrics
    um = uncorrected.metrics
    return {
        "case_id": label,
        "regime": regime,
        "route_generation_method": _route_method(method),
        "method": method,
        "variant": variant,
        "path": "realistic" if method == "holographic" and variant == "lab" else "ideal",
        "beam_air_zone_um": beam["metrics"]["canonical_zone_um"],
        "sample_uncorrected_zone_um": um["canonical_zone_um"],
        "sample_corrected_zone_um": cm["canonical_zone_um"],
        "strict_bessel_region_um": cm["strict_bessel_region_um"],
        "sample_corrected_peak_z_um": cm["peak_z_um"],
        "ring_or_core_um": cm["ring_radius_um"] if int(cm["ell"]) else cm["core_radius_um"],
        "peak_fluence_J_cm2": cm["peak_fluence_J_cm2"],
        "side_to_core_peak_ratio": cm["side_to_core_peak_ratio"],
        "medium_n": cm["medium_n"],
        "surface_transmission": cm["surface_transmission"],
        "interface_correction_label_uncorrected": um["interface_correction_label"],
        "interface_correction_label_corrected": cm["interface_correction_label"],
        "interface_correction_implementation": cm["interface_correction_implementation"],
        "corrected_rel_l2_to_no_interface": cm["corrected_rel_l2_to_no_interface"],
        "uncorrected_rel_l2_to_no_interface": cm["uncorrected_rel_l2_to_no_interface"],
        "phase_only_power_relative_error": cm["phase_only_power_relative_error"],
        "spherical_after_waves": cm["interface_spherical_after_waves"],
        **winding,
    }


In [5]:
rows = []
correction_rows = []
results = {}
for regime in ("general", "limits"):
    for method in ("holographic", "physical"):
        for variant in ("ideal", "lab"):
            cfg, path = configured_case(method, regime, variant)
            label = f"{regime}_{method}_{variant}"
            beam = bt.run_case(cfg, preset=PRESET, path=path, case_id=f"{label}_beam")
            uncorrected = vbb_sample_study.run_through_sample(beam["surface_field"], cfg, correct_interface=False)
            corrected = vbb_sample_study.run_through_sample(beam["surface_field"], cfg, correct_interface=True)
            winding = winding_fields(beam, method, cfg)
            rows.append(row_for(label, regime, method, variant, beam, uncorrected, corrected, winding))
            for sample in (uncorrected, corrected):
                sm = sample.metrics
                correction_rows.append({
                    "case_id": f"{label}_{sm['interface_correction_label']}",
                    "regime": regime,
                    "method": method,
                    "variant": variant,
                    "path": path,
                    "route_generation_method": _route_method(method),
                    "canonical_zone_um": sm["canonical_zone_um"],
                    "strict_bessel_region_um": sm["strict_bessel_region_um"],
                    "interface_correction_label": sm["interface_correction_label"],
                    "interface_correction_implementation": sm["interface_correction_implementation"],
                    "corrected_rel_l2_to_no_interface": sm["corrected_rel_l2_to_no_interface"],
                    "uncorrected_rel_l2_to_no_interface": sm["uncorrected_rel_l2_to_no_interface"],
                    "phase_only_power_relative_error": sm["phase_only_power_relative_error"],
                    "medium_n": sm["medium_n"],
                    "surface_transmission": sm["surface_transmission"],
                    **winding,
                })
            results[label] = {"beam": beam, "uncorrected": uncorrected, "corrected": corrected}

summary = pd.DataFrame(rows)
summary = lab_schema.with_lab_realism_metadata(
    summary,
    generation_method="interface_corrected_numerical",
    model_level="interface_model",
    hardware_status="diagnostic_only",
    plane_label="in_medium_plane",
    coordinate_frame="sample_plane_um_z_from_surface_in_medium",
    run_id=RUN_ID,
    preset=PRESET,
    path="through_sample",
)
summary.to_csv(out_csv / "through_sample_summary.csv", index=False)

correction_rows_stamped = []
for row in correction_rows:
    label = row["interface_correction_label"]
    lab_schema.annotate_lab_realism_row(
        row,
        generation_method="interface_corrected_numerical" if label == "ideal_numerical_correction" else "interface_uncorrected",
        model_level="interface_model",
        hardware_status="diagnostic_only" if label == "ideal_numerical_correction" else _route_hardware_status(row["method"], row["variant"]),
        plane_label="in_medium_plane",
        coordinate_frame="sample_plane_um_z_from_surface_in_medium",
        run_id=RUN_ID,
        preset=PRESET,
        path=row["path"],
    )
    correction_rows_stamped.append(row)
interface_correction_summary = lab_schema.ordered_lab_realism_frame(correction_rows_stamped)
interface_correction_summary.to_csv(out_csv / "interface_correction_summary.csv", index=False)
corrected_vs_uncorrected_metrics = interface_correction_summary.copy()
corrected_vs_uncorrected_metrics.to_csv(out_csv / "corrected_vs_uncorrected_metrics.csv", index=False)
summary


,run_id,generated_at_utc,source_schema_version,case_id,preset,path,generation_method,model_level,hardware_status,plane_label,...,side_to_core_peak_ratio,medium_n,surface_transmission,interface_correction_label_uncorrected,interface_correction_label_corrected,interface_correction_implementation,corrected_rel_l2_to_no_interface,uncorrected_rel_l2_to_no_interface,phase_only_power_relative_error,spherical_after_waves
0,20260715T180107Z,2026-07-15T18:01:07.431120+00:00,1.0.0,general_holographic_ideal,fast,ideal,interface_corrected_numerical,interface_model,diagnostic_only,in_medium_plane,...,0.429350,2.44,0.96,uncorrected_interface,ideal_numerical_correction,diagnostic_only,0.007046,1.350925,1.773750e-16,-0.000089
1,20260715T180107Z,2026-07-15T18:01:07.431120+00:00,1.0.0,general_holographic_lab,fast,realistic,interface_corrected_numerical,interface_model,diagnostic_only,in_medium_plane,...,0.331575,2.44,0.96,uncorrected_interface,ideal_numerical_correction,diagnostic_only,0.007087,1.357514,1.592704e-16,-0.000161
2,20260715T180107Z,2026-07-15T18:01:07.431120+00:00,1.0.0,general_physical_ideal,fast,ideal,interface_corrected_numerical,interface_model,diagnostic_only,in_medium_plane,...,0.411231,2.44,0.96,uncorrected_interface,ideal_numerical_correction,diagnostic_only,0.007038,1.355268,0.000000e+00,-0.000089
3,20260715T180107Z,2026-07-15T18:01:07.431120+00:00,1.0.0,general_physical_lab,fast,ideal,interface_corrected_numerical,interface_model,diagnostic_only,in_medium_plane,...,0.411401,2.44,0.96,uncorrected_interface,ideal_numerical_correction,diagnostic_only,0.007038,1.355321,0.000000e+00,-0.000089
4,20260715T180107Z,2026-07-15T18:01:07.431120+00:00,1.0.0,limits_holographic_ideal,fast,ideal,interface_corrected_numerical,interface_model,diagnostic_only,in_medium_plane,...,0.500058,2.44,0.96,uncorrected_interface,ideal_numerical_correction,diagnostic_only,0.007084,1.424083,0.000000e+00,-0.000089
5,20260715T180107Z,2026-07-15T18:01:07.431120+00:00,1.0.0,limits_holographic_lab,fast,realistic,interface_corrected_numerical,interface_model,diagnostic_only,in_medium_plane,...,1.000000,2.44,0.96,uncorrected_interface,ideal_numerical_correction,diagnostic_only,0.007907,1.101286,0.000000e+00,-0.000161
6,20260715T180107Z,2026-07-15T18:01:07.431120+00:00,1.0.0,limits_physical_ideal,fast,ideal,interface_corrected_numerical,interface_model,diagnostic_only,in_medium_plane,...,0.460143,2.44,0.96,uncorrected_interface,ideal_numerical_correction,diagnostic_only,0.007231,1.405722,1.879174e-16,-0.000089
7,20260715T180107Z,2026-07-15T18:01:07.431120+00:00,1.0.0,limits_physical_lab,fast,ideal,interface_corrected_numerical,interface_model,diagnostic_only,in_medium_plane,...,0.459999,2.44,0.96,uncorrected_interface,ideal_numerical_correction,diagnostic_only,0.007231,1.405752,0.000000e+00,-0.000089


In [6]:
# All 8 computed cases: holographic + physical × general + limits × ideal + lab.
# Task G: preset/N/device_downsample stated in title.
from vbb_study.viz_fields import measured_charge_label as _mcl_nb05

_N05 = base0.grid.N
_ds05 = base0.grid.device_downsample
print(f"Plotting all {len(results)} through-sample cases (preset={PRESET}, N={_N05}, device_downsample={_ds05})")
for label in sorted(results.keys()):
    beam = results[label]["beam"]
    sf = beam["surface_field"]
    design = beam["design"]
    sample_radius_m = float(design.vortex_main_ring_radius_m)
    method = "physical" if "physical" in label else "holographic"
    conj_mode = "preserve_vortex" if method == "physical" else None
    charge_lbl = _mcl_nb05(
        sf.Ex, sf.grid, sample_radius_m,
        design_ell=int(design.ell), conjugate_mode=conj_mode,
    )
    title = (
        f"Through-sample {label.replace('_', ' ')}\n"
        f"{charge_lbl} | preset={PRESET}, N={_N05}, device_downsample={_ds05}"
    )
    vbb_sample_study.plot_sample_result_comparison(
        results[label]["uncorrected"],
        results[label]["corrected"],
        out_fig / f"through_sample_{label}.png",
        title=title,
    )
    print(f"  {label}: {charge_lbl}")

Plotting all 8 through-sample cases (preset=fast, N=512, device_downsample=4)


  general_holographic_ideal: measured winding = 3.00 (design ℓ=3; ✓ charge preserved)


  general_holographic_lab: measured winding = 3.00 (design ℓ=3; ✓ charge preserved)


  general_physical_ideal: measured winding = 3.00 (design ℓ=3; ✓ charge preserved)


  general_physical_lab: measured winding = 3.00 (design ℓ=3; ✓ charge preserved)


  limits_holographic_ideal: measured winding = 3.00 (design ℓ=3; ✓ charge preserved)


  limits_holographic_lab: measured winding = -1.00 (design ℓ=3; charge stripped)


  limits_physical_ideal: measured winding = 3.00 (design ℓ=3; ✓ charge preserved)


  limits_physical_lab: measured winding = 3.00 (design ℓ=3; ✓ charge preserved)
